# 02 — Titoli puliti e testate solo politiche (tentativo 11, esperimenti del weekend)

Due varianti del dataset di `01`, fissate il 23 ottobre (venerdì) prima dei risultati.

**1. Titolo + articolo.** I titoli scaricati spesso contengono il **nome del sito** ("... - 2020 NEWS", "Breitbart |
..."): un modello imparerebbe a riconoscere la testata, non a leggere il titolo. Per ogni testata si spezzano i titoli
sui separatori ` - `, ` – `, ` — `, ` | `, ` • `, ` » `, ` :: `; un pezzo **iniziale o finale** che torna in almeno il
20% dei titoli della testata (e almeno 3 volte) è una cornice del sito e si toglie. Uscita: `Dataset/versioni/titoli.csv`
(`article_id`, `titolo_pulito`).

**2. Solo politici.** Le testate che per NewsGuard 2026 hanno come argomento (`Topics`) **solo** "Political news or
commentary": via quelle con più generi (per esempio politica + salute + complotti, la rete di Natural News).
Training e test solo su queste; stessi fold di `01`; il peso si ricalcola dentro il sottoinsieme (min(1, 5/n), n =
testate del gruppo che restano). Uscita: `Dataset/versioni/solo_politici.csv`.

In [1]:
import re
from collections import Counter

import pandas as pd

from comune import LIVELLI, NG2026, RESULTS_DATASET, VERSIONI, leggi_newsguard

SEP = re.compile(r"\s+(?:[-–—|•»]|::)\s+")
art = pd.read_csv(VERSIONI / "dataset.csv", usecols=["article_id", "domain", "titolo", "valutazione"])
art["titolo"] = art.titolo.fillna("").astype(str).str.strip()
t = pd.read_csv(VERSIONI / "fold.csv").set_index("domain")

## 1. I titoli puliti

In [2]:
cornici = {}
for dom, g in art.groupby("domain"):
    pezzi = [SEP.split(x) for x in g.titolo if x]
    n = len(pezzi)
    fine = Counter(p[-1].strip().lower() for p in pezzi if len(p) > 1)
    inizio = Counter(p[0].strip().lower() for p in pezzi if len(p) > 1)
    cornici[dom] = ({s for s, c in fine.items() if c >= 3 and c >= 0.2 * n}, {s for s, c in inizio.items() if c >= 3 and c >= 0.2 * n})


def pulisci(titolo, dom):
    fine, inizio = cornici.get(dom, (set(), set()))
    if not titolo:
        return titolo
    p = SEP.split(titolo); n = len(p)
    while len(p) > 1 and p[-1].strip().lower() in fine:
        p = p[:-1]
    while len(p) > 1 and p[0].strip().lower() in inizio:
        p = p[1:]
    return titolo if len(p) == n else " - ".join(x.strip() for x in p)       # cambia solo se si è tolta una cornice


art["titolo_pulito"] = [pulisci(x, d) for x, d in zip(art.titolo, art.domain)]
cambiati = art.titolo != art.titolo_pulito
print(f"titoli: {len(art)}; vuoti: {(art.titolo == '').sum()}; puliti (cornice tolta): {cambiati.sum()} ({cambiati.mean():.0%}); "
      f"testate con una cornice: {sum(1 for f, i in cornici.values() if f or i)} su {len(cornici)}")
art[["article_id", "titolo_pulito"]].to_csv(VERSIONI / "titoli.csv", index=False)
art[cambiati].groupby("domain").head(1)[["domain", "titolo", "titolo_pulito"]].head(15)

titoli: 71218; vuoti: 1; puliti (cornice tolta): 14615 (21%); testate con una cornice: 127 su 835


,domain,titolo,titolo_pulito
769,africanvoiceonline.co.uk,Nigeria at 66: The Digital Transformation Nige...,Nigeria at 66: The Digital Transformation Nige...
958,afrotech.com,"Xavier Ochsner College Of Medicine, The Nation...","Xavier Ochsner College Of Medicine, The Nation..."
1160,aier.org,"Policy Briefing: Federal Spending, Debt, and F...","Policy Briefing: Federal Spending, Debt, and F..."
1436,alphanews.org,Minneapolis father and son plead guilty to $9 ...,Minneapolis father and son plead guilty to $9 ...
2351,americanpress.com,Jim Beam column:Trump plays ball with Putin - ...,Jim Beam column:Trump plays ball with Putin
2768,americaoutloud.com,"Karl Bushby: The man who walked 31,000 miles a...","Karl Bushby: The man who walked 31,000 miles a..."
2968,americustimesrecorder.com,Ethan Wolfe gives vision for Georgia House Dis...,Ethan Wolfe gives vision for Georgia House Dis...
3277,amnews.com,Forkland Festival celebrates 55 years of tradi...,Forkland Festival celebrates 55 years of tradi...
3648,anthonyblogan.com,Mike Tyson Confirms He Delivered a Letter from...,Mike Tyson Confirms He Delivered a Letter from...
4040,asianlite.com,China dumps US debt as India’s Treasury holdin...,China dumps US debt as India’s Treasury holdin...


## 2. Le testate solo politiche

In [3]:
n26 = leggi_newsguard(NG2026)
argomenti = t.index.map(n26.Topics).fillna("").str.split(",").map(lambda x: [a.strip() for a in x if a.strip()])
t["argomenti"] = argomenti.map(", ".join)
solo = argomenti.map(lambda x: x == ["Political news or commentary"]).to_numpy()
pol = t[solo].copy()
pol["testate_nel_gruppo"] = pol.groupby("gruppo").label.transform("size")
pol["peso"] = (5 / pol.testate_nel_gruppo).clip(upper=1.0)
pol.reset_index()[["domain", "gruppo", "valutazione", "peso", "fold_gruppo", "label", "label_2022"]].to_csv(VERSIONI / "solo_politici.csv", index=False)
righe = []
for nome, sel in [("valutate", pol.valutazione), ("tutte (training)", pol.valutazione | True)]:
    for etichetta in ["label", "label_2022"]:
        r = {"testate": nome, "etichetta": "2026" if etichetta == "label" else "2022", "n": int(sel.sum()), "gruppi": int(pol.gruppo[sel].nunique()),
             "affidabili": float((pol[etichetta][sel] >= 2).mean())}
        r.update(pol[etichetta][sel].map(dict(enumerate(LIVELLI))).value_counts().reindex(LIVELLI).fillna(0).astype(int).to_dict())
        righe.append(r)
comp = pd.DataFrame(righe).set_index(["testate", "etichetta"]); comp.to_csv(RESULTS_DATASET / "02_composizione_solo_politici.csv")
print("argomenti delle testate tolte (le più frequenti):"); print(t[~solo].argomenti.value_counts().head(6).to_string())
print(f"\narticoli delle testate solo politiche: {int(art.domain.isin(pol.index).sum())}")
comp.round(3)

argomenti delle testate tolte (le più frequenti):
argomenti
Health or medical information, Political news or commentary, Conspiracy theories or hoaxes    351
Local News, General News                                                                       53
Political news or commentary, Business or Personal Finance                                     45
Local News                                                                                     45
Political news or commentary, General News                                                     20
Conspiracy theories or hoaxes, Political news or commentary                                    13

articoli delle testate solo politiche: 25403


n  gruppi  affidabili  l1  l2  l3  l4  l5
testate          etichetta                                             
valutate         2026       143     110       0.615  17  38  22  45  21
                 2022       143     110       0.587  26  33  21  46  17
tutte (training) 2026       187     133       0.551  21  63  24  55  24
                 2022       187     133       0.540  42  44  23  56  22